## Current Final Connectome Density Distribution By Diagnosis

This section reads only the live final connectome folder: `data/derivatives/connectomes`. It computes the current `fd_sum` matrix density directly from the files that downstream analysis sees, then summarizes and plots CN, MCI, and AD separately. It does not read QC manifests.


In [ ]:
# LIVE_FINAL_DENSITY_DISTRIBUTION
from pathlib import Path
import re
import numpy as np
import pandas as pd

try:
    import plotly.express as px
except Exception as exc:
    px = None
    print(f"Plotly is unavailable in this kernel: {exc}")

PROJECT_ROOT_LIVE = Path("/home/ec2-user/exp")
LIVE_CONNECTOME_DIR = PROJECT_ROOT_LIVE / "data" / "derivatives" / "connectomes"
COHORT_CSV = PROJECT_ROOT_LIVE / "cohort" / "dti.csv"

EXPECTED_AAL3_GAPS = {35, 36, 81, 82}
VALID_AAL3_ORIGINAL_LABELS = [x for x in range(1, 171) if x not in EXPECTED_AAL3_GAPS]
GROUP_ORDER = ["CN", "MCI", "AD"]
GROUP_MAP = {
    "NORMAL": "CN",
    "NL": "CN",
    "CONTROL": "CN",
    "CN": "CN",
    "SMC": "CN",
    "MCI": "MCI",
    "EMCI": "MCI",
    "LMCI": "MCI",
    "AD": "AD",
    "DEMENTIA": "AD",
}


def _sample_id_from_matrix_path(path: Path) -> str:
    name = path.name
    match = re.match(r"SC_AAL_(.+)_fd_sum\.csv$", name)
    return match.group(1) if match else name.replace("SC_AAL_", "").replace("_fd_sum.csv", "")


def _subject_id(sample_id: str) -> str:
    match = re.search(r"\d{3}_S_\d{4}", sample_id)
    return match.group(0) if match else sample_id


def _image_id(sample_id: str) -> str:
    match = re.search(r"I\d+", sample_id)
    return match.group(0) if match else ""


def _matrix_density(path: Path) -> dict:
    arr = np.genfromtxt(path, delimiter=",")
    if arr.ndim != 2 or arr.shape[0] != arr.shape[1]:
        return {
            "n_rows": int(arr.shape[0]) if arr.ndim >= 1 else 0,
            "n_cols": int(arr.shape[1]) if arr.ndim == 2 else 0,
            "present_density": np.nan,
            "valid_zero_rows": np.nan,
            "finite_ok": False,
        }
    arr = np.nan_to_num(arr.astype(float), nan=0.0, posinf=0.0, neginf=0.0)
    n = arr.shape[0]
    if n >= 170:
        idx = [label - 1 for label in VALID_AAL3_ORIGINAL_LABELS if (label - 1) < n]
        work = arr[np.ix_(idx, idx)]
    else:
        work = arr
    tri = np.triu_indices(work.shape[0], 1)
    denom = len(tri[0])
    density = float(np.count_nonzero(work[tri]) / denom) if denom else np.nan
    valid_zero_rows = int(np.sum(np.isclose(work.sum(axis=1), 0.0)))
    return {
        "n_rows": int(n),
        "n_cols": int(n),
        "present_density": density,
        "valid_zero_rows": valid_zero_rows,
        "finite_ok": bool(np.isfinite(arr).all()),
    }


def _load_group_map() -> pd.DataFrame:
    if not COHORT_CSV.exists():
        return pd.DataFrame(columns=["subject_id", "image_id", "group"])
    cohort = pd.read_csv(COHORT_CSV)
    group_col = next((c for c in ["Research Group", "Group", "DX", "diagnosis"] if c in cohort.columns), None)
    subject_col = next((c for c in ["Subject ID", "Subject", "subject_id", "RID"] if c in cohort.columns), None)
    image_col = next((c for c in ["Image ID", "image_id", "IMAGEUID"] if c in cohort.columns), None)
    if group_col is None or subject_col is None:
        return pd.DataFrame(columns=["subject_id", "image_id", "group"])
    out = pd.DataFrame()
    out["subject_id"] = cohort[subject_col].astype(str).str.extract(r"(\d{3}_S_\d{4})", expand=False).fillna(cohort[subject_col].astype(str))
    if image_col:
        out["image_id"] = "I" + cohort[image_col].astype(str).str.replace(r"^I", "", regex=True).str.replace(r"\.0$", "", regex=True)
    else:
        out["image_id"] = ""
    out["group"] = cohort[group_col].astype(str).str.upper().str.strip().replace(GROUP_MAP)
    out = out[out["group"].isin(GROUP_ORDER)].drop_duplicates(["subject_id", "image_id", "group"])
    return out


fd_sum_paths = sorted(LIVE_CONNECTOME_DIR.glob("SC_AAL_*_fd_sum.csv"))
records = []
for path in fd_sum_paths:
    sample_id = _sample_id_from_matrix_path(path)
    row = {
        "sample_id": sample_id,
        "subject_id": _subject_id(sample_id),
        "image_id": _image_id(sample_id),
        "matrix_path": str(path),
    }
    row.update(_matrix_density(path))
    records.append(row)

live_density_df = pd.DataFrame(records)
groups = _load_group_map()
if not live_density_df.empty and not groups.empty:
    exact = live_density_df.merge(groups, on=["subject_id", "image_id"], how="left")
    matched = exact[exact["group"].notna()].copy()
    missing = exact[exact["group"].isna()].drop(columns=["group"])

    # Use subject-only fallback only when the cohort has one unambiguous diagnosis for that subject.
    subject_group_counts = groups.groupby("subject_id")["group"].nunique()
    unambiguous_subjects = subject_group_counts[subject_group_counts.eq(1)].index
    fallback_groups = (
        groups[groups["subject_id"].isin(unambiguous_subjects)]
        .drop_duplicates("subject_id")[["subject_id", "group"]]
    )
    fallback = missing.merge(fallback_groups, on="subject_id", how="left")
    live_density_df = pd.concat([matched, fallback], ignore_index=True)
else:
    live_density_df["group"] = "UNKNOWN"

live_density_df["group"] = live_density_df["group"].fillna("UNKNOWN")
plot_df = live_density_df[live_density_df["group"].isin(GROUP_ORDER)].copy()
plot_df["group"] = pd.Categorical(plot_df["group"], categories=GROUP_ORDER, ordered=True)
plot_df = plot_df.sort_values(["group", "sample_id"])

summary = (
    plot_df.groupby("group", observed=True)
    .agg(
        n=("sample_id", "nunique"),
        mean_density=("present_density", "mean"),
        median_density=("present_density", "median"),
        q1_density=("present_density", lambda s: s.quantile(0.25)),
        q3_density=("present_density", lambda s: s.quantile(0.75)),
        min_density=("present_density", "min"),
        max_density=("present_density", "max"),
        median_valid_zero_rows=("valid_zero_rows", "median"),
    )
    .reset_index()
)

unknown_n = int((live_density_df["group"] == "UNKNOWN").sum())
print(f"Live final fd_sum matrices found: {len(live_density_df)}")
print(f"Grouped CN/MCI/AD matrices: {len(plot_df)}")
print(f"Ungrouped/UNKNOWN matrices: {unknown_n}")
display(summary.style.format({
    "mean_density": "{:.4f}",
    "median_density": "{:.4f}",
    "q1_density": "{:.4f}",
    "q3_density": "{:.4f}",
    "min_density": "{:.4f}",
    "max_density": "{:.4f}",
    "median_valid_zero_rows": "{:.1f}",
}))

if px is not None and not plot_df.empty:
    fig = px.violin(
        plot_df,
        x="group",
        y="present_density",
        color="group",
        box=True,
        points="all",
        hover_data=["sample_id", "subject_id", "image_id", "n_rows", "valid_zero_rows"],
        category_orders={"group": GROUP_ORDER},
        title="Current live fd_sum connectome density by diagnosis group",
        color_discrete_map={"CN": "#4DA3D9", "MCI": "#9BCB6B", "AD": "#F05A5A"},
    )
    fig.update_traces(marker={"size": 4, "opacity": 0.45}, meanline_visible=True)
    fig.update_layout(
        template="plotly_white",
        font={"color": "#111827", "size": 13},
        title_font={"color": "#111827", "size": 20},
        xaxis_title="Diagnosis group",
        yaxis_title="fd_sum density",
        legend_title_text="Group",
    )
    fig.update_xaxes(tickfont={"color": "#111827"}, title={"font": {"color": "#111827"}})
    fig.update_yaxes(tickfont={"color": "#111827"}, title={"font": {"color": "#111827"}})
    fig.show()

    hist = px.histogram(
        plot_df,
        x="present_density",
        color="group",
        facet_col="group",
        nbins=40,
        category_orders={"group": GROUP_ORDER},
        title="Current live fd_sum density histograms by diagnosis group",
        color_discrete_map={"CN": "#4DA3D9", "MCI": "#9BCB6B", "AD": "#F05A5A"},
    )
    hist.update_layout(
        template="plotly_white",
        font={"color": "#111827", "size": 13},
        title_font={"color": "#111827", "size": 20},
        xaxis_title="fd_sum density",
        yaxis_title="Subject count",
        showlegend=False,
    )
    hist.for_each_xaxis(lambda axis: axis.update(tickfont={"color": "#111827"}, title={"font": {"color": "#111827"}}))
    hist.for_each_yaxis(lambda axis: axis.update(tickfont={"color": "#111827"}, title={"font": {"color": "#111827"}}))
    hist.show()
else:
    display(plot_df[["sample_id", "group", "present_density", "valid_zero_rows", "n_rows"]])


# Structural Connectome SC-Forge

Clean contract-first notebook for the current EC2 structural-connectome cohort.

This notebook does not replace `structural_connectome_A.ipynb` or `structural_connectome_B.ipynb`. It is a clean orchestration layer for the SC-Forge design: discover the current completed connectomes, QC them, build an analysis gate, and plan reruns only for subjects that need a specific upstream lane.

Default behavior is non-destructive. Heavy execution is disabled until you explicitly change the run-control flags.

## Run Policy

- `DRY_RUN=True` and `EXECUTE=False` means commands are printed or metadata is written only.
- Existing production matrices are never overwritten by this notebook.
- Current matrix QC is safe to run because it only reads CSV matrices.
- A production rerun should happen only after a canary passes the SC-Forge gates.
- Analysis should use only `PASS` subjects, plus `WARN` only after explicit review.

In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('/home/ec2-user/exp')
PYTHON = Path('/home/ec2-user/fsl/bin/python')
SCFORGE_ROOT = PROJECT_ROOT / 'scforge'
SCFORGE_CONFIG = SCFORGE_ROOT / 'configs' / 'scforge.yaml'
DERIV_ROOT = PROJECT_ROOT / 'data' / 'derivatives'
CONNECTOMES_DIR = DERIV_ROOT / 'connectomes'
SCFORGE_DERIV = DERIV_ROOT / 'sc_forge_v1'
SCFORGE_GROUP_QC = SCFORGE_DERIV / 'group_qc'

# Run controls. Keep these conservative unless you are intentionally running a canary.
DRY_RUN = True
EXECUTE = False
WRITE_SAFE_MANIFESTS = True
RUN_CURRENT_MATRIX_QC = True
RUN_SMOKE_TESTS = False
FORCE_INVALID_ONLY = True
BACKUP_BEFORE_REPLACE = True
MAX_QC_WORKERS = 8

# Optional subject controls.
INCLUDE: list[str] = []
EXCLUDE: list[str] = []
SELECTED_SID: str | None = None

sys.path.insert(0, str(SCFORGE_ROOT))

def run_scforge(args: list[str], *, check: bool = False) -> subprocess.CompletedProcess[str]:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(SCFORGE_ROOT)
    cmd = [str(PYTHON), '-m', 'scforge.cli', *args, '--config', str(SCFORGE_CONFIG)]
    proc = subprocess.run(cmd, cwd=str(SCFORGE_ROOT), env=env, text=True, capture_output=True)
    print('$ ' + ' '.join(cmd))
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f'scforge command failed with code {proc.returncode}: {cmd}')
    return proc

def write_csv_safe(df: pd.DataFrame, path: Path, *, index: bool = False) -> None:
    if WRITE_SAFE_MANIFESTS:
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=index)
        print(f'wrote {path} rows={len(df)}')
    else:
        print(f'WRITE_SAFE_MANIFESTS=False; would write {path} rows={len(df)}')

print('SC-Forge notebook ready')
print('PROJECT_ROOT=', PROJECT_ROOT)
print('CONNECTOMES_DIR=', CONNECTOMES_DIR)
print('DRY_RUN=', DRY_RUN, 'EXECUTE=', EXECUTE)

## Current SC Matrix Density Snapshot

This table reads the live final connectome folder (`data/derivatives/connectomes`) and shows only `sample_id | present_density` for existing `fd_sum` structural-connectome matrices. It does not read QC manifests.


In [ ]:
# Current SC matrix density snapshot (fd_sum) from live final outputs only
from pathlib import Path
import re
import numpy as np
import pandas as pd

LIVE_CONNECTOME_DIR = Path("/home/ec2-user/exp/data/derivatives/connectomes")
EXPECTED_AAL3_GAPS = {35, 36, 81, 82}
VALID_AAL3_ORIGINAL_LABELS = [x for x in range(1, 171) if x not in EXPECTED_AAL3_GAPS]


def _live_sample_id(path: Path) -> str:
    match = re.match(r"SC_AAL_(.+)_fd_sum\.csv$", path.name)
    return match.group(1) if match else path.stem


def _live_density(path: Path) -> float:
    arr = np.nan_to_num(np.genfromtxt(path, delimiter=","), nan=0.0, posinf=0.0, neginf=0.0)
    if arr.ndim != 2 or arr.shape[0] != arr.shape[1]:
        return np.nan
    if arr.shape[0] >= 170:
        idx = [label - 1 for label in VALID_AAL3_ORIGINAL_LABELS if (label - 1) < arr.shape[0]]
        arr = arr[np.ix_(idx, idx)]
    tri = np.triu_indices(arr.shape[0], 1)
    return float(np.count_nonzero(arr[tri]) / len(tri[0])) if len(tri[0]) else np.nan

current_density_snapshot = pd.DataFrame(
    [
        {"sample_id": _live_sample_id(p), "present_density": _live_density(p)}
        for p in sorted(LIVE_CONNECTOME_DIR.glob("SC_AAL_*_fd_sum.csv"))
    ]
).sort_values("sample_id")

print(f"Live final fd_sum matrices found: {len(current_density_snapshot)}")
display(current_density_snapshot[["sample_id", "present_density"]].style.format({"present_density": "{:.4f}"}))


## 0. SC-Forge Package Sanity

In [ ]:
run_scforge(['status'], check=True)
run_scforge(['aal3-summary'], check=True)

if RUN_SMOKE_TESTS:
    proc = subprocess.run(
        [str(PYTHON), '-m', 'unittest', 'discover', '-s', 'tests'],
        cwd=str(SCFORGE_ROOT),
        env={**os.environ, 'PYTHONPATH': str(SCFORGE_ROOT)},
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError('SC-Forge smoke tests failed')

## 1. Initialize SC-Forge Derivative Tree

This creates only the new `sc_forge_v1` metadata tree when `EXECUTE=True`. It does not touch existing production outputs.

In [ ]:
init_args = ['init']
if EXECUTE:
    init_args.append('--execute')
run_scforge(init_args, check=True)

## 2. Discover Current EC2 Connectome Cohort

This is the practical limited cohort we can evaluate now. A subject is discovered from files named `SC_AAL_<SID>_<metric>.csv` under `/home/ec2-user/exp/data/derivatives/connectomes`.

In [ ]:
MATRIX_METRICS = [
    'count',
    'count_invnodevol',
    'fd_sum',
    'len_mean',
    'invlen_mean',
    'fa_mean',
    'md_mean',
    'rd_mean',
    'ad_mean',
    'ALL',
]
metric_regex = '|'.join(re.escape(m) for m in sorted(MATRIX_METRICS, key=len, reverse=True))
pattern = re.compile(rf'^SC_AAL_(?P<sid>.+)_(?P<metric>{metric_regex})\.csv$')

records = []
for path in sorted(CONNECTOMES_DIR.glob('SC_AAL_*.csv')):
    match = pattern.match(path.name)
    if not match:
        continue
    records.append({'sid': match.group('sid'), 'metric': match.group('metric'), 'path': str(path), 'size_bytes': path.stat().st_size})

matrix_inventory = pd.DataFrame(records)
if INCLUDE:
    matrix_inventory = matrix_inventory[matrix_inventory['sid'].isin(INCLUDE)].copy()
if EXCLUDE:
    matrix_inventory = matrix_inventory[~matrix_inventory['sid'].isin(EXCLUDE)].copy()

display(matrix_inventory.head())
print('matrix files:', len(matrix_inventory))
print('subjects with any matrix:', matrix_inventory['sid'].nunique() if len(matrix_inventory) else 0)

availability = matrix_inventory.assign(present=1).pivot_table(
    index='sid', columns='metric', values='present', aggfunc='max', fill_value=0
).reset_index()
for metric in MATRIX_METRICS:
    if metric not in availability.columns:
        availability[metric] = 0
availability['has_required_all'] = availability[['count', 'fd_sum', 'len_mean', 'fa_mean', 'md_mean', 'rd_mean', 'ad_mean']].min(axis=1).astype(int)
availability['has_primary'] = availability[['count', 'fd_sum']].min(axis=1).astype(int)

summary = availability[MATRIX_METRICS + ['has_required_all', 'has_primary']].sum().to_frame('n_subjects').reset_index(names='metric')
display(summary)

write_csv_safe(matrix_inventory, SCFORGE_GROUP_QC / 'current_connectome_matrix_inventory.csv')
write_csv_safe(availability, SCFORGE_GROUP_QC / 'current_connectome_subject_availability.csv')

## 3. Cohort Group Linkage

This links discovered matrix subjects to CN/MCI/AD labels where possible. It uses the local `cohort/dti.csv` and tries common subject/group column names.

In [ ]:
def _normalise_subject_code(value: object) -> str:
    match = re.search(r'\d{3}_S_\d{4}', str(value))
    return match.group(0) if match else ''


def _normalise_image_id(value: object) -> str:
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none'}:
        return ''
    if re.fullmatch(r'\d+(\.0)?', text):
        text = text.split('.')[0]
    match = re.search(r'I?(\d+)', text, flags=re.IGNORECASE)
    return f"I{match.group(1)}" if match else ''


def _normalise_full_sid(value: object) -> str:
    match = re.search(r'\d{3}_S_\d{4}_I\d+', str(value))
    return match.group(0) if match else ''


def _normalise_group(value: object) -> str:
    text = str(value).strip().upper()
    mapping = {
        'CONTROL': 'CN',
        'NORMAL': 'CN',
        'NL': 'CN',
        'COGNITIVELY NORMAL': 'CN',
        'CN': 'CN',
        'MCI': 'MCI',
        'EMCI': 'MCI',
        'LMCI': 'MCI',
        'AD': 'AD',
        'DEMENTIA': 'AD',
        'SMC': 'SMC',
    }
    return mapping.get(text, text if text in {'CN', 'MCI', 'AD', 'SMC'} else 'UNKNOWN')


cohort_paths = [PROJECT_ROOT / 'cohort' / 'dti.csv', PROJECT_ROOT / 'cohort' / 'dti_master.csv']
cohort = pd.DataFrame()
cohort_tables = []
for cohort_path in cohort_paths:
    if cohort_path.exists():
        part = pd.read_csv(cohort_path)
        part['_cohort_source'] = cohort_path.name
        cohort_tables.append(part)
if cohort_tables:
    cohort_raw = pd.concat(cohort_tables, ignore_index=True, sort=False)
    discovered = set(availability['sid'].astype(str))
    candidate_group_cols = [c for c in cohort_raw.columns if c.lower() in {'group', 'diagnosis', 'dx', 'research group', 'research_group'}]
    group_col = candidate_group_cols[0] if candidate_group_cols else None

    best_sid = pd.Series('', index=cohort_raw.index, dtype=object)
    best_source = 'none'
    best_score = 0

    for col in cohort_raw.columns:
        full_sid = cohort_raw[col].map(_normalise_full_sid)
        score = len(set(full_sid[full_sid.astype(bool)]).intersection(discovered))
        if score > best_score:
            best_sid, best_source, best_score = full_sid, col, score

    subject_cols = [c for c in cohort_raw.columns if cohort_raw[c].map(_normalise_subject_code).astype(bool).sum() > 0]
    image_cols = [c for c in cohort_raw.columns if ('image' in c.lower() or 'id' in c.lower())]
    for subj_col in subject_cols:
        subject_code = cohort_raw[subj_col].map(_normalise_subject_code)
        for image_col in image_cols:
            if image_col == subj_col:
                continue
            image_id = cohort_raw[image_col].map(_normalise_image_id)
            combined = subject_code.where(subject_code.astype(bool), '') + '_' + image_id.where(image_id.astype(bool), '')
            combined = combined.where(subject_code.astype(bool) & image_id.astype(bool), '')
            score = len(set(combined[combined.astype(bool)]).intersection(discovered))
            if score > best_score:
                best_sid, best_source, best_score = combined, f'{subj_col}+{image_col}', score

    print(f"cohort files={[p.name for p in cohort_paths if p.exists()]} sid_source={best_source} matched={best_score}/{len(discovered)} group_col={group_col}")
    cohort = cohort_raw.copy()
    cohort['sid'] = best_sid
    cohort = cohort[cohort['sid'].astype(bool)].copy()
    cohort['group'] = cohort[group_col].map(_normalise_group) if group_col else 'UNKNOWN'
    cohort = cohort[['sid', 'group']].drop_duplicates('sid')
    if best_score == 0:
        print('WARNING: no cohort rows matched discovered SC matrix subject IDs')
else:
    print('missing cohort files:', cohort_paths)

availability_with_group = availability.merge(cohort, on='sid', how='left') if len(cohort) else availability.assign(group='UNKNOWN')
availability_with_group['group'] = availability_with_group['group'].fillna('UNKNOWN')
display(availability_with_group.groupby('group')[['has_primary', 'has_required_all']].sum().astype(int))
write_csv_safe(availability_with_group, SCFORGE_GROUP_QC / 'current_connectome_subject_availability_with_group.csv')

## 4. AAL3 Atlas Contract

AAL3 is the thesis atlas. Expected original-label gaps are not treated as failures. AAL_007 and AAL_008 are required valid frontal opercular labels.

In [ ]:
node_table_path = SCFORGE_GROUP_QC / 'aal3_node_table.tsv'
if WRITE_SAFE_MANIFESTS:
    run_scforge(['write-aal3-node-table', str(node_table_path)], check=True)
    aal3_node_table = pd.read_csv(node_table_path, sep='\t')
    display(aal3_node_table.head(12))
    print('AAL3 nodes:', len(aal3_node_table))
else:
    print('WRITE_SAFE_MANIFESTS=False; skipping node table write')

## 5. Matrix QC For Current Completed Connectomes

This reads existing `fd_sum` matrices. It does not overwrite anything. Strict gate: readable, square, finite, symmetric, zero diagonal, density >= configured minimum, and no unexpected valid zero rows.

In [ ]:
from scforge.config import load_config
from scforge.qc import matrix_qc_from_atlas_config

cfg = load_config(SCFORGE_CONFIG)
fd_sum_paths = matrix_inventory[matrix_inventory['metric'] == 'fd_sum'][['sid', 'path']].copy()
print('fd_sum matrices to QC:', len(fd_sum_paths))

def qc_one(row: dict) -> dict:
    result = matrix_qc_from_atlas_config(row['path'], cfg.qc, cfg.atlas).to_dict()
    result['sid'] = row['sid']
    return result

qc_records = []
if RUN_CURRENT_MATRIX_QC and len(fd_sum_paths):
    with ThreadPoolExecutor(max_workers=MAX_QC_WORKERS) as pool:
        futures = [pool.submit(qc_one, row._asdict() if hasattr(row, '_asdict') else {'sid': row.sid, 'path': row.path}) for row in fd_sum_paths.itertuples(index=False)]
        for future in as_completed(futures):
            qc_records.append(future.result())
    current_matrix_qc = pd.DataFrame(qc_records).sort_values('sid')
else:
    current_matrix_qc = pd.DataFrame()

if len(current_matrix_qc):
    current_matrix_qc = current_matrix_qc.merge(availability_with_group[['sid', 'group', 'has_required_all']], on='sid', how='left')
    current_matrix_qc['analysis_gate'] = np.where(current_matrix_qc['pass_strict'].astype(bool) & (current_matrix_qc['has_required_all'] == 1), 'include', 'exclude_pending_repair')
    display(current_matrix_qc.head())
    display(current_matrix_qc.groupby(['group', 'status', 'analysis_gate']).size().reset_index(name='n'))
    display(current_matrix_qc['reasons'].value_counts().head(20).reset_index(name='n').rename(columns={'index': 'reasons'}))
    write_csv_safe(current_matrix_qc, SCFORGE_GROUP_QC / 'current_fd_sum_matrix_qc.csv')
else:
    print('Matrix QC not run or no fd_sum matrices found.')

## 6. Analysis Gate Export

This is the gate downstream analysis should use. It is based on existing outputs plus strict matrix QC. It does not claim failed subjects are irreparable; it keeps them out until a canary-proven repair succeeds.

In [ ]:
if 'current_matrix_qc' in globals() and len(current_matrix_qc):
    analysis_gate = current_matrix_qc[[
        'sid', 'group', 'analysis_gate', 'status', 'density', 'nonzero_upper_edges', 'valid_zero_rows', 'reasons', 'has_required_all'
    ]].copy()
    analysis_gate['gate_rule'] = 'PASS strict fd_sum matrix QC and all required matrices present'
    write_csv_safe(analysis_gate, SCFORGE_GROUP_QC / 'analysis_gate_scforge_current.csv')
    display(analysis_gate.groupby(['group', 'analysis_gate']).size().reset_index(name='n'))
else:
    print('Run matrix QC first.')

## 7. Select A Subject For End-to-End Planning

This section is for one selected subject at a time. It prints the SC-Forge command plan and does not run heavy jobs unless you explicitly copy the command into a tmux job or add an execution wrapper later.

In [ ]:
if SELECTED_SID is None:
    if len(availability):
        SELECTED_SID = availability.sort_values(['has_required_all', 'sid'], ascending=[False, True]).iloc[0]['sid']
print('SELECTED_SID=', SELECTED_SID)

subject_root = SCFORGE_DERIV / 'subjects' / str(SELECTED_SID)
subject_dirs = {
    'dwi': subject_root / 'dwi',
    'anat': subject_root / 'anat',
    'xfm': subject_root / 'xfm',
    'atlas': subject_root / 'atlas',
    'fod': subject_root / 'fod',
    'tractography': subject_root / 'tractography',
    'connectome': subject_root / 'connectome',
    'qc': subject_root / 'qc',
}
pd.DataFrame({'stage': list(subject_dirs), 'path': [str(p) for p in subject_dirs.values()]})

## 8. Source And DWI Planning

Fill the raw NIfTI/bval/bvec/json paths for a selected subject when testing a true upstream rerun. Existing current matrices can still be QC-gated without these paths.

In [ ]:
# Fill these when running a true source-contract or DWI preprocessing plan.
RAW_DWI_NIFTI = None
RAW_DWI_BVAL = None
RAW_DWI_BVEC = None
RAW_DWI_JSON = None

if RAW_DWI_NIFTI and RAW_DWI_BVAL and RAW_DWI_BVEC:
    source_args = [
        'source-contract', '--subject', str(SELECTED_SID),
        '--nifti', str(RAW_DWI_NIFTI), '--bval', str(RAW_DWI_BVAL), '--bvec', str(RAW_DWI_BVEC),
        '--output', str(subject_dirs['qc'] / 'source_contract.json'),
    ]
    if RAW_DWI_JSON:
        source_args.extend(['--json-sidecar', str(RAW_DWI_JSON)])
    run_scforge(source_args, check=False)
    run_scforge([
        'plan-dwi', '--subject', str(SELECTED_SID),
        '--nifti', str(RAW_DWI_NIFTI), '--bval', str(RAW_DWI_BVAL), '--bvec', str(RAW_DWI_BVEC),
        '--out-dir', str(subject_dirs['dwi'])
    ], check=True)
else:
    print('Raw DWI paths are not set. Source/DWI planning skipped for now.')

## 9. Anatomy, Atlas, 5TT, FOD, Tracks, And Connectome Planners

These planners encode the intended SC-Forge contract. They are placeholders until the selected subject has the needed source paths.

## 10. Canary Selection From Current QC

This picks a small canary panel from current matrix QC: severe failures, moderate failures, and controls. It is a planning list, not an automatic rerun.

In [ ]:
# Fill these after source/anatomy discovery for the selected subject.
T1_NATIVE = None
SUBJECTS_DIR_SUBJECT = None
T1_TO_B0_MRTRIX = None
DWI_BIASCORR_MIF = None
DWI_MASK_MIF = None
WMFOD_MIF = None
FIVE_TT_MIF = None
GMWMI_MIF = None
TRACKS_TCK = None
AAL3_NODES_B0_1MM = None
SIFT2_WEIGHTS = None
FA_MIF = None
MD_MIF = None
RD_MIF = None
AD_MIF = None

if SUBJECTS_DIR_SUBJECT and T1_NATIVE and T1_TO_B0_MRTRIX:
    run_scforge([
        'plan-5tt', '--subject', str(SELECTED_SID),
        '--subjects-dir-subject', str(SUBJECTS_DIR_SUBJECT),
        '--t1-native', str(T1_NATIVE),
        '--t1-to-b0-mrtrix', str(T1_TO_B0_MRTRIX),
        '--out-dir', str(subject_dirs['anat'])
    ], check=True)
else:
    print('5TT planner skipped: set SUBJECTS_DIR_SUBJECT, T1_NATIVE, T1_TO_B0_MRTRIX.')

if DWI_BIASCORR_MIF and DWI_MASK_MIF:
    run_scforge([
        'plan-fod', '--subject', str(SELECTED_SID),
        '--dwi-biascorr', str(DWI_BIASCORR_MIF), '--mask', str(DWI_MASK_MIF),
        '--out-dir', str(subject_dirs['fod']), '--shells', '0', '1000'
    ], check=True)
else:
    print('FOD planner skipped: set DWI_BIASCORR_MIF and DWI_MASK_MIF.')

if WMFOD_MIF and FIVE_TT_MIF and GMWMI_MIF:
    run_scforge([
        'plan-preflight', '--subject', str(SELECTED_SID),
        '--wmfod', str(WMFOD_MIF), '--five-tt', str(FIVE_TT_MIF), '--gmwmi', str(GMWMI_MIF),
        '--out-dir', str(subject_dirs['tractography'])
    ], check=True)
else:
    print('Tractography planner skipped: set WMFOD_MIF, FIVE_TT_MIF, and GMWMI_MIF.')

if TRACKS_TCK and AAL3_NODES_B0_1MM:
    args = [
        'plan-connectome', '--subject', str(SELECTED_SID),
        '--tracks', str(TRACKS_TCK), '--nodes', str(AAL3_NODES_B0_1MM),
        '--out-dir', str(subject_dirs['connectome']), '--assignment-variant', 'radial8',
    ]
    optional = {
        '--sift2-weights': SIFT2_WEIGHTS,
        '--fa-mif': FA_MIF,
        '--md-mif': MD_MIF,
        '--rd-mif': RD_MIF,
        '--ad-mif': AD_MIF,
    }
    for flag, value in optional.items():
        if value:
            args.extend([flag, str(value)])
    run_scforge(args, check=True)
else:
    print('Connectome planner skipped: set TRACKS_TCK and AAL3_NODES_B0_1MM.')

## 10. Canary Selection From Current QC

This picks a small canary panel from current matrix QC: severe failures, moderate failures, and controls. It is a planning list, not an automatic rerun.